# Python参数传递机制

## 学习目标
- 理解Python中参数传递的底层原理
- 掌握"赋值传递"的概念（既不是值传递也不是引用传递）
- 理解不可变对象和可变对象在参数传递中的行为差异
- 避免参数传递中的常见错误

---

## 1️⃣ 问题引入：为什么需要了解参数传递？

### 常见踩坑场景

```python
def modify_list(lst):
    lst.append(4) # 修改列表
return lst

my_list = [1, 2, 3]
result = modify_list(my_list)
print(my_list)  # [1, 2, 3, 4]  ← 糟了！原列表被修改了！
print(result)    # [1, 2, 3, 4]
```

**问题**：我期望 `my_list` 在函数调用后不变，但事与愿违！

这就是因为**不了解Python参数传递机制**导致的bug。花很多时间debug，最后发现是传参过程中数据结构的改变。

---

## 2️⃣ 其他语言中的参数传递

### 值传递（Pass by Value）

**特点**：拷贝参数的值，传递给函数里的新变量。原变量和新变量互相独立，互不影响。

**C++示例**：
```cpp
#include <iostream>
using namespace std;

void swap(int x, int y) {
int temp;
temp = x;
x = y;
y = temp;
}

int main() {
int a = 1;
int b = 2;
swap(a, b);
cout << "a: " << a << endl;  // 1 (不变)
cout << "b: " << b << endl;  // 2 (不变)
return 0;
}
```

**结果**：`a` 和 `b` 的值不变，因为 `swap()` 函数操作的是 **a 和 b 的拷贝**。

### 引用传递（Pass by Reference）

**特点**：把参数的引用传给新的变量，原变量和新变量指向同一块内存地址。改变其中一个，另一个也会改变。

**C++示例**：
```cpp
#include <iostream>
using namespace std;

void swap(int& x, int& y) {  // 注意这里的 &
int temp;
temp = x;
x = y;
y = temp;
}

int main() {
int a = 1;
int b = 2;
swap(a, b);
cout << "a: " << a << endl;  // 2 (改变！)
cout << "b: " << b << endl;  // 1 (改变！)
return 0;
}
```

**结果**：`a` 和 `b` 的值被交换了，因为 `x` 和 `y` 是 `a` 和 `b` 的**引用**（别名）。

---

## 3️⃣ Python中的参数传递：赋值传递

### 核心概念

**Python的参数传递既不是值传递，也不是引用传递，而是：赋值传递（Pass by Assignment）或对象的引用传递（Pass by Object Reference）**

**通俗理解**：
- 调用函数时，把**实参**赋值给**形参**（就像 `param = arg` 一样）
- 函数内部的参数变量，和函数外部的实参变量，**指向同一个对象**
- 但是，它们**是两个独立的变量**（只是指向同一个对象）

### 示例1：基本赋值

```python
def modify(x):
    print(f"Inside: x id = {id(x)}")
x = x + 1 # 重新赋值
print(f"After modify: x = {x}, id = {id(x)}")

a = 1
print(f"Before: a = {a}, id = {id(a)}")
modify(a)
print(f"After: a = {a}, id = {id(a)}")
```

**输出**：
```
Before: a = 1, id = 1407062464
Inside: x id = 1407062464   ← x 和 a 指向同一个对象（1）
After modify: x = 2, id = 1407062496  ← x 重新指向新对象（2）
After: a = 1, id = 1407062464      ← a 仍然指向原对象（1）
```

**解析**：
- `modify(a)` 相当于 `x = a`，所以 `x` 和 `a` 都指向对象 `1`
- `x = x + 1` 让 `x` 指向新对象 `2`，但 `a` 仍然指向 `1`
- **结论**：重新赋值不会影响原变量


In [1]:
def modify(x):
    print(f"Inside: x = {x}, id = {id(x)}")
    x = x + 1  # 重新赋值
    print(f"After modify: x = {x}, id = {id(x)}")

a = 1
print(f"Before: a = {a}, id = {id(a)}")
modify(a)
print(f"After: a = {a}, id = {id(a)}")
print("")
print("结论：a 没有被修改，因为 x 重新赋值指向了新对象")


Before: a = 1, id = 140711390172072
Inside: x = 1, id = 140711390172072
After modify: x = 2, id = 140711390172104
After: a = 1, id = 140711390172072

结论：a 没有被修改，因为 x 重新赋值指向了新对象


---

## 4️⃣ 不可变对象 vs 可变对象

### 核心区别

| 类型 | 举例 | 能否修改？ | 参数传递行为 |
|------|------|-----------|----------------|
| **不可变（Immutable）** | int, float, str, tuple | ❌ 不能 | 函数内修改会创建新对象，不影响原变量 |
| **可变（Mutable）** | list, dict, set |  能 | 函数内修改会影响原变量 |

### 示例2：不可变对象（int）

```python
def modify_int(x):
    print(f"Inside before: x = {x}, id = {id(x)}")
x = x + 10 # 创建新对象
print(f"Inside after: x = {x}, id = {id(x)}")

a = 5
print(f"Before: a = {a}, id = {id(a)}")
modify_int(a)
print(f"After: a = {a}, id = {id(a)}")
```

**结果**：`a` 不变，因为 `x = x + 10` 让 `x` 指向了新对象 `15`

### 示例3：可变对象（list）

```python
def modify_list(lst):
    print(f"Inside before: lst = {lst}, id = {id(lst)}")
lst.append(4) # 原地修改
print(f"Inside after: lst = {lst}, id = {id(lst)}")

my_list = [1, 2, 3]
print(f"Before: my_list = {my_list}, id = {id(my_list)}")
modify_list(my_list)
print(f"After: my_list = {my_list}, id = {id(my_list)}")
```

**结果**：`my_list` 被修改了！因为 `lst.append(4)` **原地修改**了列表对象。

**关键点**：
- `lst` 和 `my_list` 指向**同一个列表对象**
- `lst.append(4)` 修改了这个对象，所以 `my_list` 也"看"到了变化


In [2]:
def modify_list(lst):
    print(f"Inside before: lst = {lst}, id = {id(lst)}")
    lst.append(4)  # 原地修改
    print(f"Inside after: lst = {lst}, id = {id(lst)}")

# lst 和 my_list 仍然指向同一个对象（id相同）
my_list = [1, 2, 3]
print(f"Before: my_list = {my_list}, id = {id(my_list)}")
modify_list(my_list)
print(f"After: my_list = {my_list}, id = {id(my_list)}")
print("")
print("结论：my_list 被修改了，因为 lst 和 my_list 指向同一个对象")


Before: my_list = [1, 2, 3], id = 2996258632960
Inside before: lst = [1, 2, 3], id = 2996258632960
Inside after: lst = [1, 2, 3, 4], id = 2996258632960
After: my_list = [1, 2, 3, 4], id = 2996258632960

结论：my_list 被修改了，因为 lst 和 my_list 指向同一个对象


---

## 5️⃣ 通过函数改变变量的两种方法是

### 方法1：创建新变量保存修改后的值，然后返回给原变量

```python
def add_one(n):
    return n + 1 # 返回新值

a = 1
a = add_one(a) # 重新赋值
print(a) # 2
```

**适用场景**：不可变对象（int, str, tuple）

### 方法2：直接在可变对象上修改

```python
def add_element(lst, element):
    lst.append(element) # 原地修改

my_list = [1, 2, 3]
add_element(my_list, 4)
print(my_list) # [1, 2, 3, 4]
```

**适用场景**：可变对象（list, dict, set）

---

## 6️⃣ 常见错误与陷阱

### 陷阱1：期望不可变对象被修改

```python
def modify(x):
    x = x + 1 # 这不会修改原变量！

a = 1
modify(a)
print(a) # 1 (不变)
```

**错误原因**：不可变对象的"修改"会创建新对象，原变量仍然指向旧对象。

**改正**：
```python
def modify(x):
    return x + 1 # 返回新值

a = 1
a = modify(a) # 重新赋值
print(a) # 2 (改变)
```

### 陷阱2：不小心修改了可变对象

```python
def dangerous_function(lst):
    lst.clear() # 清空列表（危险！）

my_list = [1, 2, 3]
dangerous_function(my_list)
print(my_list) # [] (被清空了！)
```

**错误原因**：函数内原地修改了可变对象，影响了原变量。

**改正**：
```python
def safe_function(lst):
    new_lst = lst.copy() # 创建拷贝
new_lst.clear()
return new_lst

my_list = [1, 2, 3]
empty_list = safe_function(my_list)
print(my_list)     # [1, 2, 3] (不变)
print(empty_list)  # [] (新列表)
```


---

## 7️⃣ 变量赋值 vs 改变变量

### 变量赋值（让变量指向某个对象）

```python
a = [1, 2, 3] # 创建对象，让 a 指向它
b = a          # 让 b 也指向同一个对象
```

**结果**：`a` 和 `b` 指向同一个对象。

### 改变变量（修改对象的值）

```python
# 情况1：不可变对象（创建新对象）
a = 1
a = a + 1 # a 指向新对象 2

# 情况2：可变对象（原地修改）
lst = [1, 2, 3]
lst.append(4) # 原地修改 lst
```

### 变量可以被删除，但对象无法被删除

```python
a = [1, 2, 3]
b = a
del a # 删除变量 a
print(b) # [1, 2, 3]  ← 对象还在！
```

**解析**：
- `del a` 只是删除了变量 `a`，但对象 `[1, 2, 3]` 还在内存中（因为 `b` 还指向它）
- Python有垃圾回收机制，当对象没有被任何变量指向时，才会被回收

---

## 8️⃣ 内存图解

### 示例：lst = [1, 2, 3]

```
变量:   lst

对象:  [1, 2, 3]  (内存地址: 0x123)
```

### 示例：b = lst

```
变量:   lst, b

对象:  [1, 2, 3]  (内存地址: 0x123)
```

### 示例：函数调用 modify_list(lst)

```python
def modify_list(param):
    param.append(4)

lst = [1, 2, 3]
modify_list(lst)
```

**内存图**：
```
变量:   lst, param

对象:  [1, 2, 3, 4]  (同一个对象被修改了)
```


---

## 9️⃣ 总结与对比

### Python参数传递的核心要点

1. **Python的参数传递是赋值传递（Pass by Assignment）**
- 调用函数时，相当于把实参赋值给形参（`param = arg`）
- 形参和实参指向**同一个对象**，但它们是**两个独立的变量**

2. **不可变对象的"修改"会创建新对象**
- `x = x + 1` 让 `x` 指向新对象，不影响原变量
- 如果需要修改，应该返回新值并重新赋值

3. **可变对象的修改会影响所有指向该对象的变量**
- `lst.append(4)` 原地修改列表，所有指向该列表的变量都会"看到"变化
- 如果不想影响原变量，应该先拷贝（`lst.copy()`）

### 值传递 vs 引用传递 vs 赋值传递

| 传递方式 | 原变量是否受影响 | Python中 |
|----------|------------------|---------|
| 值传递 | ❌ 不受影响 | ❌ 不是 |
| 引用传递 |  受影响 | ❌ 不是 |
| **赋值传递** | **取决于对象是否可变** |  是 |

---

## 练习与自测

### 基础自测（概念检查）
- [ ] 能解释什么是"赋值传递"
- [ ] 能说出不可变对象和可变对象的区别
- [ ] 能解释为什么 `a = 1; modify(a)` 后 `a` 不变
- [ ] 能解释为什么 `lst = [1, 2, 3]; modify_list(lst)` 后 `lst` 改变
- [ ] 知道如何避免不小心修改可变对象

### 代码练习

#### 题目1：预测输出（⭐）

```python
def func(a, b):
    a.append(4)
b = b + 1
return a, b

x = [1, 2, 3]
y = 5
result_x, result_y = func(x, y)

print("x:", x)     # [1,2,3,4]
print("y:", y)     # 5
print("result_x:", result_x) # [1,2,3,4]
print("result_y:", result_y) # 6
```

**任务**：先自己思考输出结果，然后运行代码验证。

---

#### 题目2：写一个安全的函数（⭐⭐）

**题目**：
写一个函数 `add_element_safe(lst, element)`，实现：
- 向列表添加元素
- 不影响原列表
- 返回新列表

**你的代码**：


In [3]:
def func(a, b):
    a.append(4)
    b = b + 1
    return a, b

x = [1, 2, 3]
y = 5
result_x, result_y = func(x, y)

print("x:", x)
print("y:", y)
print("result_x:", result_x)
print("result_y:", result_y)

x: [1, 2, 3, 4]
y: 5
result_x: [1, 2, 3, 4]
result_y: 6


In [4]:
import copy

def add_element_safe(lst, element):
    """
    向列表添加元素，不影响原列表
    :param lst: 原列表
    :param element: 要添加的元素
    :return: 新列表
    """
    new_lst = copy.deepcopy(lst)
    new_lst.append(element)
    return new_lst

# 测试
original = [1, 2, 3]
new_list = add_element_safe(original, 4)
print("original:", original)  # 应该输出 [1, 2, 3]
print("new_list:", new_list)  # 应该输出 [1, 2, 3, 4]

original: [1, 2, 3]
new_list: [1, 2, 3, 4]


---

#### 题目3：实现一个真正的交换函数（⭐⭐⭐）

**题目**：
Python中，你不能像C++那样直接交换两个变量（因为赋值传递）。
请写一个函数，实现交换两个变量的值。

**提示**：Python中可以返回多个值。

**你的代码**：


In [5]:
def swap(a, b):
    """
    交换两个变量的值
    :return: (b, a)
    """
    new_a = b
    new_b = a
    return new_a, new_b

# 测试
x, y = 1, 2
x, y = swap(x, y)
print(f"x = {x}, y = {y}")  # 应该输出 x = 2, y = 1

x = 2, y = 1


---

## 知识总结

### 核心要点

1. **Python的参数传递是赋值传递**
- 调用函数时，实参赋值给形参
- 形参和实参指向同一个对象，但它们是两个独立的变量

2. **不可变对象（int, str, tuple）**
- "修改"时会创建新对象
- 函数内修改不会影响原变量
- 如果需要修改，应该返回新值并重新赋值

3. **可变对象（list, dict, set）**
- 可以原地修改
- 函数内修改会影响所有指向该对象的变量
- 如果不想影响原变量，应该先拷贝

4. **避免bug的最佳实践**
- 函数内尽量不修改传入的可变对象
- 如果需要修改，先拷贝（`lst.copy()` 或 `dict.copy()`）
- 或者在文档中明确说明会修改传入的对象

### 与其他知识的关系

- **前置知识**：变量赋值、对象的概念
- **后续知识**：深拷贝浅拷贝（解决可变对象拷贝问题）、闭包（变量作用域）

---

## 扩展阅读

1. **官方文档 - 参数传递**
https://docs.python.org/3/faq/programming.html#how-do-i-write-a-function-with-output-parameters-passing-a-large-mutable-object-in-by-reference

2. **了解更多**
- 为什么Python设计成这样？（历史原因、简单性）
- 其他语言（Java、JavaScript）的参数传递机制

3. **相关面试题**
- Python中参数传递是值传递还是引用传递？（答案：赋值传递）
- 以下代码的输出是什么？（可变对象vs不可变对象）

---

## 学习检查（学完自测）

- [ ] 能不看资料，默写出参数传递的示例代码
- [ ] 能独立完成所有练习
- [ ] 能解释给"小白"听（费曼学习法）
- [ ] 能说出实际工作中如何避免参数传递的bug

---

**Next**: 下一讲：装饰器 → 用装饰器优雅地修改函数行为！ 
